# Notebook 2: Temporal Framing and Labels

**Purpose:** Implement problem framing, core data cleaning, and label generation logic on the sampled data.

**Actions:**
- Ingest sampled Parquet files, apply type-casting, and filter invalid rows.
- Enrich transactions with article and customer dimension data.
- Establish the temporal framework (`snap_date` cutoffs and 7-day forward label windows).
- Generate positive labels (purchases in the 7-day forward window) and sample window-aware negatives.

*Note: This notebook uses Pandas for data manipulation and analysis.*

## 1. Setup and Imports

In this section, we import the necessary Python libraries for data manipulation and set the random seed for reproducibility.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

for _nb in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (_nb / "utils").is_dir():
        sys.path[:0] = [str(_nb)]
        break

from utils.data_casting import save_labeled_data

# Set random seed for reproducibility
np.random.seed(42)

## 2. Define Data Paths

Next, we define the file paths for the input sampled dataset (Parquet files) and the output directory.

In [2]:
def find_repo_root() -> Path:
    """Locate repository root (contains configs/). Works from repo root or notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / "notebooks"):
        if (candidate / "configs").is_dir():
            return candidate
    return Path.cwd()


ACTIVE_DATASET = os.getenv("ACTIVE_DATASET", "sample_2000_users")
REPO_ROOT = find_repo_root()


def get_data_paths(dataset_name: str = ACTIVE_DATASET) -> dict:
    """
    Define and return the paths for the input and output datasets.

    Args:
        dataset_name: Subfolder under dataset/ (e.g. ``dummy`` or ``sample_2000_users``).
            Override via ACTIVE_DATASET env var for smoke tests.

    Returns:
        dict: Paths to input Parquet tables and labeled-data output directory.
    """
    base = REPO_ROOT / "dataset" / dataset_name
    out_dir = base / "transactions_with_label"

    paths = {
        "articles": str(base / "articles"),
        "customers": str(base / "customers"),
        "transactions": str(base / "transactions"),
        "output": str(out_dir),
        "active_dataset": dataset_name,
    }
    return paths

data_paths = get_data_paths()
os.makedirs(data_paths["output"], exist_ok=True)
print(f"Active dataset: {data_paths['active_dataset']}")
print(f"Output directory ready: {data_paths['output']}")

Active dataset: sample_2000_users
Output directory ready: F:\git-projects\fashion-recommendation-system\dataset\sample_2000_users\transactions_with_label


## 3. Load Sampled Data

We load the sampled articles, customers, and transactions data from Parquet files into Pandas DataFrames.

In [3]:
def load_data(paths: dict) -> tuple:
    """
    Load the sampled articles, customers, and transactions data from Parquet files.
    
    Args:
        paths (dict): Dictionary containing data paths.
        
    Returns:
        tuple: A tuple containing (articles_df, customers_df, transactions_df).
    """
    print("Loading articles...")
    articles = pd.read_parquet(paths["articles"])
    
    print("Loading customers...")
    customers = pd.read_parquet(paths["customers"])
    
    print("Loading transactions...")
    transactions = pd.read_parquet(paths["transactions"])
    
    return articles, customers, transactions

articles, customers, transactions = load_data(data_paths)
print("Data loaded successfully.")

Loading articles...
Loading customers...
Loading transactions...
Data loaded successfully.


## 4. Schema Normalization & Null Handling

We normalize the schema and handle missing values across the three tables:
- **Transactions:** Parse dates and drop rows with null join keys.
- **Customers:** Cap ages between 16 and 100, and impute missing ages with the median.
- **Articles:** Standardize string columns to lowercase and strip whitespace.

In [4]:
def normalize_schema(articles: pd.DataFrame, customers: pd.DataFrame, transactions: pd.DataFrame) -> tuple:
    """
    Normalize schema and handle null values.
    - Parse dates in transactions
    - Cap ages (16-100) and impute missing ages in customers
    - Standardize strings in articles
    - Drop null join keys
    
    Args:
        articles (pd.DataFrame): Articles dataframe.
        customers (pd.DataFrame): Customers dataframe.
        transactions (pd.DataFrame): Transactions dataframe.
        
    Returns:
        tuple: Normalized (articles, customers, transactions).
    """
    # 1. Transactions
    transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
    transactions.dropna(subset=['customer_id', 'article_id'], inplace=True)
    
    # 2. Customers
    customers.dropna(subset=['customer_id'], inplace=True)
    median_age = customers['age'].median()
    customers['age'] = customers['age'].fillna(median_age)
    customers['age'] = customers['age'].clip(lower=16, upper=100).astype(int)
    
    # 3. Articles
    articles.dropna(subset=['article_id'], inplace=True)
    cat_cols = ['garment_group_name', 'product_type_name', 'colour_group_name', 'index_group_name']
    for col in cat_cols:
        if col in articles.columns:
            articles[col] = articles[col].astype(str).str.strip().str.lower()
            
    return articles, customers, transactions

articles, customers, transactions = normalize_schema(articles, customers, transactions)
print("Schema normalization complete.")

Schema normalization complete.


## 5. Outliers & Validity

We clean the transactions data by addressing outliers and duplicates:
- Drop transactions with negative or zero prices.
- Winsorize extreme prices at the 99th percentile to reduce the impact of outliers.
- Deduplicate exact repeating rows.

In [5]:
def handle_outliers_and_validity(transactions: pd.DataFrame) -> pd.DataFrame:
    """
    Handle outliers and validity in transactions.
    - Drop negative prices
    - Winsorize extreme prices at 99th percentile
    - Deduplicate transactions
    
    Args:
        transactions (pd.DataFrame): Transactions dataframe.
        
    Returns:
        pd.DataFrame: Cleaned transactions dataframe.
    """
    # Drop negative or zero prices
    transactions = transactions[transactions['price'] > 0].copy()
    
    # Winsorize extreme prices (99th percentile)
    p99 = transactions['price'].quantile(0.99)
    transactions['price'] = transactions['price'].clip(upper=p99)
    
    # Deduplicate exact repeats
    transactions.drop_duplicates(subset=['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id'], inplace=True)
    
    return transactions

transactions = handle_outliers_and_validity(transactions)
print("Outliers handled and transactions deduplicated.")

Outliers handled and transactions deduplicated.


## 6. Dimension Enrichment

We enrich the transactions by joining them with the articles and customers tables. This step also acts as an orphan filter, dropping any transactions that lack matching records in the dimension tables.

In [6]:
def enrich_dimensions(transactions: pd.DataFrame, articles: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    """
    Inner-join transactions to articles and customers to drop orphans and enrich data.
    
    Args:
        transactions (pd.DataFrame): Transactions dataframe.
        articles (pd.DataFrame): Articles dataframe.
        customers (pd.DataFrame): Customers dataframe.
        
    Returns:
        pd.DataFrame: Enriched transactions dataframe.
    """
    # Select only necessary columns to avoid memory bloat
    art_cols = ['article_id', 'garment_group_name', 'product_type_name', 'colour_group_name', 'index_group_name']
    art_cols = [c for c in art_cols if c in articles.columns]
    
    cust_cols = ['customer_id', 'age']
    cust_cols = [c for c in cust_cols if c in customers.columns]
    
    # Inner join drops orphan transactions
    enriched = transactions.merge(articles[art_cols], on='article_id', how='inner')
    enriched = enriched.merge(customers[cust_cols], on='customer_id', how='inner')
    
    return enriched

enriched_transactions = enrich_dimensions(transactions, articles, customers)
print(f"Enriched transactions shape: {enriched_transactions.shape}")

Enriched transactions shape: (43868, 12)


## 7. Temporal Framing & Labels

We establish the temporal framework using predefined `snap_date` cutoffs. For each snap date:
- **Positives:** Purchases made by a user in the 7-day forward window.
- **Negatives:** Sampled window-aware negatives (1:5 ratio), ensuring we exclude items the user has already seen or purchased in the window.

In [7]:
def generate_temporal_labels(transactions: pd.DataFrame) -> pd.DataFrame:
    """
    Establish snap_date cutoffs and generate positive/negative labels.
    - Define snap dates
    - Generate positives (purchases in 7-day forward window)
    - Sample window-aware negatives (1:5 ratio, exclude seen items)
    
    Args:
        transactions (pd.DataFrame): Enriched transactions dataframe.
        
    Returns:
        pd.DataFrame: Dataset with labels and snap_dates.
    """
    # Define snap dates based on the feature engineering guide
    snap_dates = [
        '2020-03-24', '2020-03-31', '2020-04-07', '2020-04-14', 
        '2020-04-28', '2020-05-15', '2020-05-31', '2020-06-30', 
        '2020-07-31', '2020-08-31', '2020-09-15'
    ]
    
    all_labeled_data = []
    all_articles = transactions['article_id'].unique()
    
    for snap_date_str in snap_dates:
        snap_date = pd.to_datetime(snap_date_str)
        window_end = snap_date + pd.Timedelta(days=7)
        
        # History up to snap_date
        history = transactions[transactions['t_dat'] <= snap_date]
        
        # Purchases in the 7-day forward window (Positives)
        window_purchases = transactions[(transactions['t_dat'] > snap_date) & (transactions['t_dat'] <= window_end)]
        
        if window_purchases.empty:
            continue
            
        # Create Positives
        positives = window_purchases[['customer_id', 'article_id']].drop_duplicates()
        positives['label'] = 1
        positives['snap_date'] = snap_date
        
        # Create Negatives (1:5 ratio)
        negatives_list = []
        seen_items_per_cust = history.groupby('customer_id')['article_id'].apply(set).to_dict()
        window_items_per_cust = window_purchases.groupby('customer_id')['article_id'].apply(set).to_dict()
        
        for cust_id in positives['customer_id'].unique():
            seen = seen_items_per_cust.get(cust_id, set())
            window_seen = window_items_per_cust.get(cust_id, set())
            exclude_items = seen.union(window_seen)
            
            available_negatives = np.setdiff1d(all_articles, list(exclude_items))
            
            n_pos = len(window_seen)
            n_neg_to_sample = n_pos * 5
            
            if len(available_negatives) >= n_neg_to_sample:
                sampled_negs = np.random.choice(available_negatives, size=n_neg_to_sample, replace=False)
            else:
                sampled_negs = available_negatives
                
            neg_df = pd.DataFrame({
                'customer_id': cust_id,
                'article_id': sampled_negs,
                'label': 0,
                'snap_date': snap_date
            })
            negatives_list.append(neg_df)
            
        if negatives_list:
            negatives = pd.concat(negatives_list, ignore_index=True)
            snap_data = pd.concat([positives, negatives], ignore_index=True)
            all_labeled_data.append(snap_data)
            
    if not all_labeled_data:
        return pd.DataFrame(columns=['customer_id', 'article_id', 'label', 'snap_date'])

    final_labeled_df = pd.concat(all_labeled_data, ignore_index=True)
    return final_labeled_df

print("Generating temporal labels (this may take a moment)...")
labeled_dataset = generate_temporal_labels(enriched_transactions)
print(f"Label generation complete. Total rows: {len(labeled_dataset)}")

Generating temporal labels (this may take a moment)...


Label generation complete. Total rows: 28122


## 8. Save Processed Data

Finally, we save the generated labeled dataset to the specified output directory in Parquet format for use in downstream feature engineering and model training.

In [8]:
# save_labeled_data writes Hive Parquet partitioned by snap_date (notebook 03 stages to s3/).
saved_path = save_labeled_data(labeled_dataset, data_paths["output"])
print(f"Saved labeled data to {saved_path}")

Saved labeled data to F:\git-projects\fashion-recommendation-system\dataset\sample_2000_users\transactions_with_label
